# Bayesian Belief Robot Localization in a Simulated Maze (with Heatmaps)

**Objective.** Implement discrete **Bayesian filtering** (a.k.a. Markov localization) on a grid maze. At each time step we update a belief distribution over robot locations given an action and a sensor observation; we visualize the belief as a heat map so one can see it evolve over time.

**Model.** Hidden state $X_t$ is the grid cell (excluding walls). Given action $a_{t-1}$ and observation $o_t$:
1. **Prediction:** $\bar{b}_t(j) = \sum_i p(X_t{=}j \mid X_{t-1}{=}i, a_{t-1})\, b_{t-1}(i)$
2. **Update:** $b_t(j) = \eta\, p(o_t \mid X_t{=}j)\, \bar{b}_t(j)$, with $\eta$ a normalizer.

We provide a simple motion model with slippage and a discrete color sensor with noise. Beliefs are displayed as heat maps over the maze (walls in black).

## 0. Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)
print("\nNVIDIA SMI (if present):")
_sh("nvidia-smi || true")

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1. Imports, Seeding, and Helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

np.set_printoptions(precision=4, suppress=True)
SEED = 123
rng = np.random.default_rng(SEED)

ACTIONS = ['N','E','S','W']
delta = {'N':(-1,0), 'E':(0,1), 'S':(1,0), 'W':(0,-1)}

def show_heatmap(belief_grid, title="Belief Heatmap"):
    fig = plt.figure(figsize=(4,4))
    plt.imshow(belief_grid, interpolation='nearest')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 2. Maze and Sensor Map

We define a small maze with walls (1) and free cells (0). Each free cell has a **color label** `0..K-1`. The observation is a noisy color reading.

In [ ]:
@dataclass
class Maze:
    grid: np.ndarray   # 0=free, 1=wall
    colors: np.ndarray # integer color labels (meaningful only at free cells)

def make_maze():
    grid = np.array([
        [0,0,0,0,0,0,0],
        [0,1,1,0,1,1,0],
        [0,0,0,0,0,0,0],
        [0,1,0,1,0,1,0],
        [0,0,0,0,0,0,0]
    ], dtype=int)
    H, W = grid.shape
    colors = np.zeros_like(grid)
    for i in range(H):
        for j in range(W):
            if grid[i,j]==0:
                colors[i,j] = (2*i + j) % 4  # simple pattern with 4 colors
    return Maze(grid=grid, colors=colors)

maze = make_maze()
H, W = maze.grid.shape
N = H*W
free_mask = (maze.grid == 0)
free_states = np.where(free_mask.reshape(-1))[0]
num_free = len(free_states)
n_colors = int(maze.colors.max() + 1)
print("Maze:", H, "x", W, "| free cells:", num_free, "| colors:", n_colors)

def idx(i,j,W): return i*W + j
def ij(s,W): return divmod(s, W)

def belief_to_grid(b):
    g = np.zeros((H,W), dtype=float)
    g[free_mask] = b[free_mask.reshape(-1)]
    return g

## 3. Motion Model

Given action $a \in \{N,E,S,W\}$, with probability `p_success` we move to the intended neighbor if free; with probability `p_slip` we move to a uniformly random **other** available neighbor; otherwise we **stay in place**. Attempts to move into walls leave the agent in place.

In [ ]:
def neighbors(i, j, maze):
    nbrs = {}
    for a in ACTIONS:
        di, dj = delta[a]
        ni, nj = i+di, j+dj
        if 0 <= ni < H and 0 <= nj < W and maze.grid[ni,nj]==0:
            nbrs[a] = (ni, nj)
    return nbrs

def transition_matrix_for_action(a, maze, p_success=0.8, p_slip=0.15):
    T = np.zeros((N,N), dtype=float)
    for s in range(N):
        i, j = ij(s, W)
        if maze.grid[i,j]==1:
            T[s,s] = 1.0
            continue
        nbrs = neighbors(i, j, maze)
        stay = 1.0
        if a in nbrs:
            t = idx(*nbrs[a], W)
            T[s,t] += p_success
            stay -= p_success
        others = [d for d in ACTIONS if d != a and d in nbrs]
        if len(others) > 0:
            slip_each = p_slip / len(others)
            for d in others:
                t = idx(*nbrs[d], W)
                T[s,t] += slip_each
            stay -= p_slip
        T[s,s] += max(0.0, stay)
        # normalize numerical noise
        rs = T[s].sum()
        if rs > 0: T[s] /= rs
        else: T[s,s] = 1.0
    return T

def build_Ts(maze, p_success=0.8, p_slip=0.15):
    return {a: transition_matrix_for_action(a, maze, p_success, p_slip) for a in ACTIONS}

T_by_action = build_Ts(maze)

## 4. Observation Model (Discrete Colors)

If the true cell has color `c`, the sensor reports `c` with probability `sensor_correct`, and a uniformly random *other* color otherwise.

In [ ]:
def emission_vector(obs_color, maze, sensor_correct=0.85):
    e = np.zeros(N, dtype=float)
    K = int(maze.colors.max() + 1)
    wrong = max(1, K-1)
    for s in free_states:
        i, j = ij(s, W)
        c = maze.colors[i,j]
        if obs_color == c:
            e[s] = sensor_correct
        else:
            e[s] = (1.0 - sensor_correct) / wrong
    return e

## 5. Bayesian Filtering (Prediction + Update)

We iterate prediction and update for a sequence of actions and observations. Beliefs are normalized probability vectors over cells.

In [ ]:
def normalize(b, eps=1e-12):
    s = b.sum()
    return b / (s + eps)

def bayes_filter(actions, observations, T_by_action, sensor_correct, b0):
    beliefs = [normalize(b0.copy())]
    for a, o in zip(actions, observations):
        pred = T_by_action[a].T @ beliefs[-1]   # prediction
        e = emission_vector(o, maze, sensor_correct)
        post = normalize(e * pred)              # update
        beliefs.append(post)
    return beliefs

## 6. Simulate a Trajectory and Observations

We sample a random start in a free cell, then follow a hand-crafted action sequence with stochastic motion and noisy observations.

In [ ]:
def random_free_cell():
    idxs = np.argwhere(maze.grid==0)
    i, j = idxs[rng.integers(0, len(idxs))]
    return int(i), int(j)

def generate_actions(T_len):
    base = ['E']*3 + ['S']*2 + ['W']*2 + ['S']*1 + ['E']*2 + ['N']*2 + ['E']*1
    if len(base) >= T_len: return base[:T_len]
    extra = rng.choice(ACTIONS, size=T_len-len(base)).tolist()
    return base + extra

def simulate(maze, actions, p_success=0.8, p_slip=0.15, sensor_correct=0.85):
    Ts = build_Ts(maze, p_success, p_slip)
    i0, j0 = random_free_cell()
    s = idx(i0, j0, W)
    true_states = [s]
    observations = []
    for a in actions:
        probs = Ts[a][s]
        s = int(rng.choice(np.arange(N), p=probs))
        true_states.append(s)
        ii, jj = ij(s, W)
        c = maze.colors[ii, jj]
        if rng.random() < sensor_correct:
            o = int(c)
        else:
            choices = [d for d in range(int(maze.colors.max()+1)) if d != c]
            o = int(rng.choice(choices))
        observations.append(o)
    return true_states, observations, Ts

# Parameters
T_len = 16
sensor_correct = 0.85
p_success = 0.8
p_slip = 0.15

actions = generate_actions(T_len)
true_states, observations, T_by_action_sim = simulate(maze, actions, p_success, p_slip, sensor_correct)

print("Actions:", actions)
print("Observations (colors):", observations)
print("True path length:", len(true_states))

## 7. Run the Filter and Visualize Beliefs

In [ ]:
# Prior: uniform over free cells
b0 = np.zeros(N, dtype=float)
b0[free_states] = 1.0 / len(free_states)

beliefs = bayes_filter(actions, observations, T_by_action_sim, sensor_correct, b0)
print("Computed beliefs for T+1 =", len(beliefs))

for t, b in enumerate(beliefs):
    show_heatmap(belief_to_grid(b), title=f"Belief Heatmap — t={t}")

## 8. Optional: Fixed-Interval Smoothing

Given the full observation sequence, one can compute smoothed marginals $p(X_t \mid o_{1:T}) \propto b_t \odot \beta_t$, where $\beta_t$ are backward messages. This typically sharpens the later beliefs.

In [ ]:
def backward_messages(actions, observations, T_by_action, sensor_correct=0.85):
    T = len(actions)
    betas = [None]*(T+1)
    betas[T] = np.ones(N, dtype=float)
    for t in range(T-1, -1, -1):
        a = actions[t]
        o = observations[t]
        e = emission_vector(o, maze, sensor_correct)
        betas[t] = T_by_action[a] @ (e * betas[t+1])
        betas[t] = normalize(betas[t])
    return betas

def smooth(beliefs, betas):
    T = len(beliefs)-1
    sm = [None]*(T+1)
    for t in range(T+1):
        sm[t] = normalize(beliefs[t] * betas[t])
    return sm

betas = backward_messages(actions, observations, T_by_action_sim, sensor_correct)
smoothed = smooth(beliefs, betas)

show_heatmap(belief_to_grid(beliefs[-1]), title="Filtering Belief at T")
show_heatmap(belief_to_grid(smoothed[-1]), title="Smoothed Belief at T")

## 9. Qualitative Comparison to Ground Truth

We print the true $(i,j)$ positions and the MAP cell at each time according to the filtering belief.

In [ ]:
def argmax_cell(b):
    return int(np.argmax(b))

true_ij = [ij(s, W) for s in true_states]
map_cells = [argmax_cell(b) for b in beliefs]
map_ij = [ij(s, W) for s in map_cells]
print("True positions (i,j):", true_ij)
print("MAP positions (i,j):", map_ij)

## 10. Save Artifacts & Download

We persist the maze, actions, observations, filtering and smoothed beliefs, and a few PNG heatmaps. Use the helper below to download a ZIP in Colab.

In [ ]:
import os, json
os.makedirs("artifacts", exist_ok=True)

np.savez("artifacts/bayes_localization_run.npz",
         grid=maze.grid, colors=maze.colors,
         actions=np.array(actions, dtype='<U1'),
         observations=np.array(observations, dtype=int),
         beliefs=np.array(beliefs),
         smoothed=np.array(smoothed),
         true_states=np.array(true_states, dtype=int))

def save_heat(grid_b, fname, title):
    fig = plt.figure(figsize=(4,4))
    plt.imshow(grid_b, interpolation='nearest')
    plt.title(title); plt.axis('off'); plt.tight_layout()
    fig.savefig(f"artifacts/{fname}", dpi=120); plt.close(fig)

save_heat(belief_to_grid(beliefs[0]), "belief_t0.png", "Belief t=0")
mid = len(beliefs)//2
save_heat(belief_to_grid(beliefs[mid]), f"belief_t{mid}.png", f"Belief t={mid}")
save_heat(belief_to_grid(beliefs[-1]), "belief_tT.png", "Belief t=T")

print("Saved artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 11. Extensions & Research Ideas

- **Different sensors**: e.g., distance-to-walls in each compass direction (Gaussian noise), bearing to landmarks.
- **Orientation-aware state**: expand the state space to (x,y,heading) and adapt the motion model.
- **Sparse transitions**: exploit sparsity for larger mazes; vectorize updates.
- **Adaptive models**: learn sensor or motion noise parameters from data via EM.
- **Active localization**: choose actions to minimize expected entropy of the belief.